# EE 451: Communications Systems
## Lesson 26 — Sampling & Quantization

### Learning Objectives
By the end of this lesson, you will be able to:
- Apply the Nyquist sampling theorem to determine minimum sampling rate
- Explain aliasing and design anti-aliasing filters
- Analyze quantization noise and Signal-to-Quantization-Noise Ratio (SQNR)
- Design Pulse Code Modulation (PCM) systems
- Explain companding (μ-law and A-law) for voice compression

### Textbook Reference
Haykin & Moher, Chapter 5

In [ ]:
# Setup: Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2

print("Setup complete! NumPy version:", np.__version__)

## Part 1: Nyquist Sampling Theorem

**Nyquist-Shannon Sampling Theorem:** A bandlimited signal with maximum
frequency $B$ Hz can be perfectly reconstructed from its samples if the
sampling rate satisfies:

$$f_s \geq 2B \quad \text{(Nyquist rate)}$$

- $f_s$: sampling rate (samples/sec)
- $f_N = 2B$: Nyquist rate (minimum)
- $f_s / 2$: Nyquist frequency (max recoverable frequency)

In [ ]:
# === Part 1: Sampling at Different Rates ===

f_sig = 5.0     # Signal frequency (Hz)
B = f_sig        # Bandwidth
f_nyquist = 2 * B  # Nyquist rate = 10 Hz

# Continuous signal (high sample rate for plotting)
t_cont = np.linspace(0, 1, 10000)
x_cont = np.cos(2 * np.pi * f_sig * t_cont)

# Sample at different rates
sample_rates = [4, 10, 20, 50]  # Hz (undersample, Nyquist, 2x, 5x)
labels = ['f_s = 4 Hz (UNDER)', f'f_s = {int(f_nyquist)} Hz (Nyquist)',
          'f_s = 20 Hz (2\u00d7)', 'f_s = 50 Hz (5\u00d7)']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for idx, (fs, label) in enumerate(zip(sample_rates, labels)):
    ax = axes[idx // 2, idx % 2]
    
    # Sample points
    n_samples = int(fs * 1) + 1
    t_samp = np.arange(n_samples) / fs
    x_samp = np.cos(2 * np.pi * f_sig * t_samp)
    
    # Reconstruct using sinc interpolation
    x_recon = np.zeros_like(t_cont)
    for n in range(len(t_samp)):
        x_recon += x_samp[n] * np.sinc(fs * (t_cont - t_samp[n]))
    
    ax.plot(t_cont, x_cont, 'C0-', linewidth=1.5, alpha=0.4, label='Original')
    ax.stem(t_samp, x_samp, linefmt='C1-', markerfmt='C1o', basefmt='C1-',
            label=f'Samples ({fs} Hz)')
    if fs >= f_nyquist:
        ax.plot(t_cont, x_recon, 'C2--', linewidth=1.5, label='Reconstructed')
    else:
        ax.plot(t_cont, x_recon, 'r--', linewidth=1.5, label='Reconstructed (ALIASED)')
    
    ax.set_title(label, fontsize=12, fontweight='bold',
                 color='red' if fs < f_nyquist else 'black')
    ax.set_xlabel('Time (s)' if idx >= 2 else '')
    ax.set_ylabel('Amplitude')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(0, 1)
    ax.set_ylim(-1.5, 1.5)

plt.suptitle(f'Sampling a {f_sig:.0f} Hz Signal (Nyquist rate = {f_nyquist:.0f} Hz)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Nyquist Sampling Theorem:")
print(f"  Signal frequency: f = {f_sig} Hz")
print(f"  Nyquist rate: f_N = 2B = {f_nyquist} Hz")
print(f"  f_s = 4 Hz < f_N \u2192 ALIASING (reconstructed signal is wrong)")
print(f"  f_s = {int(f_nyquist)} Hz = f_N \u2192 Minimum rate (borderline)")
print(f"  f_s = 20, 50 Hz > f_N \u2192 Perfect reconstruction")

## Part 2: Aliasing Demonstration

**Aliasing** occurs when $f_s < 2B$: high-frequency components "fold back"
and appear as lower frequencies.

**Alias frequency:** If a signal at frequency $f$ is sampled at $f_s < 2f$,
it appears as $f_{alias} = |f - k \cdot f_s|$ for the nearest integer $k$.

**Frequency-domain view:** Sampling creates spectral copies at $\pm n f_s$.
If copies overlap, aliasing distortion results.

In [ ]:
# === Part 2: Aliasing in Frequency Domain ===

f_sig = 7.0   # Hz
fs = 10.0     # Sampling rate (undersampled! fs < 2*f_sig)
f_alias = abs(f_sig - fs)  # = 3 Hz

t_cont = np.linspace(0, 2, 10000)
x_orig = np.cos(2 * np.pi * f_sig * t_cont)
x_alias = np.cos(2 * np.pi * f_alias * t_cont)

# Sample
t_samp = np.arange(0, 2, 1/fs)
x_samp = np.cos(2 * np.pi * f_sig * t_samp)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Time domain
axes[0].plot(t_cont, x_orig, 'C0', linewidth=1.5, label=f'Original: {f_sig} Hz')
axes[0].plot(t_cont, x_alias, 'r--', linewidth=1.5, label=f'Alias: {f_alias} Hz')
axes[0].stem(t_samp, x_samp, linefmt='k-', markerfmt='ko', basefmt='k-',
             label=f'Samples (f_s = {fs:.0f} Hz)')
axes[0].set_xlabel('Time (s)', fontsize=13)
axes[0].set_ylabel('Amplitude', fontsize=13)
axes[0].set_title(f'Aliasing: {f_sig:.0f} Hz Signal Sampled at {fs:.0f} Hz \u2192 Appears as {f_alias:.0f} Hz',
                  fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].set_xlim(0, 1)

# Frequency domain: show spectral copies
freqs_show = np.linspace(-25, 25, 1000)
# Original spectrum: impulses at +/- f_sig
# After sampling: copies at f_sig + n*fs

axes[1].set_xlim(-25, 25)
axes[1].set_ylim(0, 1.3)

# Draw spectral copies
for n in range(-3, 4):
    f_pos = f_sig + n * fs
    f_neg = -f_sig + n * fs
    color = 'C0' if n == 0 else 'C1'
    alpha = 1.0 if n == 0 else 0.4
    lw = 3 if n == 0 else 1.5
    if -25 < f_pos < 25:
        axes[1].plot([f_pos, f_pos], [0, 1], color=color, linewidth=lw, alpha=alpha)
        axes[1].plot(f_pos, 1, '^', color=color, markersize=8, alpha=alpha)
    if -25 < f_neg < 25:
        axes[1].plot([f_neg, f_neg], [0, 1], color=color, linewidth=lw, alpha=alpha)
        axes[1].plot(f_neg, 1, '^', color=color, markersize=8, alpha=alpha)

# Mark Nyquist zone
axes[1].axvspan(-fs/2, fs/2, alpha=0.1, color='green')
axes[1].text(0, 1.15, f'Nyquist zone\n[\u2212{fs/2:.0f}, +{fs/2:.0f}] Hz',
            ha='center', fontsize=10, color='green', fontweight='bold')

# Mark alias
axes[1].annotate(f'Alias at {f_alias:.0f} Hz', xy=(f_alias, 1), xytext=(f_alias + 5, 1.1),
                fontsize=11, color='red', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='red'))

axes[1].set_xlabel('Frequency (Hz)', fontsize=13)
axes[1].set_ylabel('Magnitude', fontsize=13)
axes[1].set_title('Spectrum After Sampling (spectral copies overlap \u2192 aliasing)',
                  fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Aliasing Example:")
print(f"  Signal: f = {f_sig} Hz")
print(f"  Sampling rate: f_s = {fs} Hz")
print(f"  Nyquist rate: 2f = {2*f_sig} Hz > f_s \u2192 UNDERSAMPLED")
print(f"  Alias frequency: |f - f_s| = |{f_sig} - {fs}| = {f_alias} Hz")
print(f"\nPractical examples:")
print(f"  Wagon wheel effect (movie): wheel appears to spin backwards")
print(f"  Moir\u00e9 patterns: spatial aliasing in images")

## Part 3: Quantization and SQNR

**Quantization** maps continuous amplitude values to discrete levels.

- $L$ levels $\Rightarrow$ $b = \log_2(L)$ bits per sample
- Step size: $\Delta = (x_{max} - x_{min}) / L$
- Quantization error: $e_q = x - x_q$, uniformly distributed in $[-\Delta/2, \Delta/2]$
- **SQNR** (Signal-to-Quantization-Noise Ratio):

$$\text{SQNR (dB)} \approx 6.02b + 1.76 \text{ dB}$$

**Rule of thumb:** Each additional bit adds ~6 dB of SQNR.

In [ ]:
# === Part 3: Quantization at Different Bit Depths ===

# Generate a test signal
fs_quant = 1000
t_q = np.linspace(0, 0.1, fs_quant)
x_q = 0.8 * np.sin(2 * np.pi * 50 * t_q) + 0.3 * np.sin(2 * np.pi * 120 * t_q)

def quantize(x, n_bits, x_min=-1, x_max=1):
    L = 2**n_bits
    delta = (x_max - x_min) / L
    x_clipped = np.clip(x, x_min, x_max - delta)
    x_quantized = np.round((x_clipped - x_min) / delta) * delta + x_min + delta/2
    q_error = x - x_quantized
    return x_quantized, q_error, delta, L

bit_depths = [2, 4, 8, 16]
fig, axes = plt.subplots(len(bit_depths), 2, figsize=(14, 10))

sqnr_measured = []
sqnr_theory = []

for idx, bits in enumerate(bit_depths):
    xq, err, delta, L = quantize(x_q, bits)
    
    # SQNR
    sig_power = np.mean(x_q**2)
    noise_power = np.mean(err**2)
    sqnr_meas = 10 * np.log10(sig_power / noise_power) if noise_power > 0 else 100
    sqnr_theo = 6.02 * bits + 1.76
    sqnr_measured.append(sqnr_meas)
    sqnr_theory.append(sqnr_theo)
    
    # Signal + quantized
    axes[idx, 0].plot(t_q * 1000, x_q, 'C0', linewidth=1, alpha=0.6, label='Original')
    axes[idx, 0].step(t_q * 1000, xq, 'C1', linewidth=1.5, where='mid', label='Quantized')
    # Show quantization levels
    if bits <= 4:
        for lev in np.arange(-1 + delta/2, 1, delta):
            axes[idx, 0].axhline(lev, color='gray', linewidth=0.3, alpha=0.5)
    axes[idx, 0].set_ylabel(f'{bits}-bit\n(L={L})', fontsize=11, fontweight='bold')
    axes[idx, 0].legend(fontsize=8, loc='upper right')
    axes[idx, 0].set_ylim(-1.3, 1.3)
    
    # Quantization error
    axes[idx, 1].plot(t_q * 1000, err, 'C3', linewidth=0.8)
    axes[idx, 1].axhline(delta/2, color='red', linestyle='--', alpha=0.5, linewidth=0.8)
    axes[idx, 1].axhline(-delta/2, color='red', linestyle='--', alpha=0.5, linewidth=0.8)
    axes[idx, 1].set_ylabel(f'Error\n\u0394={delta:.4f}', fontsize=10)
    axes[idx, 1].text(95, 0, f'SQNR = {sqnr_meas:.1f} dB', fontsize=10,
                      fontweight='bold', ha='right', va='center',
                      bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

axes[-1, 0].set_xlabel('Time (ms)', fontsize=12)
axes[-1, 1].set_xlabel('Time (ms)', fontsize=12)
axes[0, 0].set_title('Signal and Quantized Version', fontsize=13, fontweight='bold')
axes[0, 1].set_title('Quantization Error', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("SQNR: Theory vs Measured")
print(f"{'Bits':<6} {'Levels':<8} {'\u0394':<12} {'Theory (dB)':<14} {'Measured (dB)'}")
print("-" * 55)
for bits, sqt, sqm in zip(bit_depths, sqnr_theory, sqnr_measured):
    L = 2**bits
    delta = 2.0 / L
    print(f"{bits:<6} {L:<8} {delta:<12.6f} {sqt:<14.2f} {sqm:.2f}")
print(f"\nRule of thumb: Each bit adds ~6 dB of SQNR")

## Part 4: SQNR vs Bit Depth

The **6 dB per bit** rule is one of the most important relationships in
digital signal processing and communications:

| Bits | Levels | SQNR (dB) | Application |
|------|--------|-----------|-------------|
| 8 | 256 | 49.9 | Telephone (μ-law) |
| 12 | 4,096 | 74.0 | Professional audio |
| 16 | 65,536 | 98.1 | CD audio |
| 24 | 16.7M | 146.2 | Studio recording |

In [ ]:
# === Part 4: SQNR vs Bit Depth ===

bits_range = np.arange(1, 25)
sqnr_formula = 6.02 * bits_range + 1.76

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# SQNR vs bits
axes[0].plot(bits_range, sqnr_formula, 'C0-o', markersize=5, linewidth=2)
axes[0].set_xlabel('Bits per Sample (b)', fontsize=13)
axes[0].set_ylabel('SQNR (dB)', fontsize=13)
axes[0].set_title('SQNR = 6.02b + 1.76 dB', fontsize=14, fontweight='bold')

# Annotate key applications
apps = [(8, 'Telephone'), (12, 'Pro Audio'), (16, 'CD'), (24, 'Studio')]
for b, app in apps:
    sqnr = 6.02 * b + 1.76
    axes[0].plot(b, sqnr, 'rs', markersize=10)
    axes[0].annotate(f'{app}\n{sqnr:.0f} dB', xy=(b, sqnr),
                     xytext=(b + 1.5, sqnr - 8), fontsize=9,
                     arrowprops=dict(arrowstyle='->', color='gray'))

# PCM bit rate vs quality
fs_options = [8000, 16000, 44100, 48000, 96000]
bit_options = [8, 16, 16, 24, 24]
labels_pcm = ['Telephone', 'Wideband', 'CD', 'DVD', 'Hi-Res']
bitrates = [f * b / 1000 for f, b in zip(fs_options, bit_options)]  # kbps
sqnrs = [6.02 * b + 1.76 for b in bit_options]

x = np.arange(len(labels_pcm))
w = 0.35
bars1 = axes[1].bar(x - w/2, bitrates, w, color='C0', alpha=0.7, label='Bit Rate')
ax2 = axes[1].twinx()
bars2 = ax2.bar(x + w/2, sqnrs, w, color='C1', alpha=0.7, label='SQNR')

axes[1].set_xticks(x)
axes[1].set_xticklabels(labels_pcm, fontsize=10)
axes[1].set_ylabel('Bit Rate (kbps)', fontsize=12, color='C0')
ax2.set_ylabel('SQNR (dB)', fontsize=12, color='C1')
axes[1].set_title('PCM System Parameters', fontsize=14, fontweight='bold')

# Combined legend
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

print("PCM System Comparison:")
print(f"{'System':<14} {'f_s (Hz)':<10} {'Bits':<6} {'Bit Rate':<12} {'SQNR (dB)'}")
print("-" * 52)
for lbl, fs, b, br in zip(labels_pcm, fs_options, bit_options, bitrates):
    sqnr = 6.02 * b + 1.76
    print(f"{lbl:<14} {fs:<10} {b:<6} {br:>8.0f} kbps {sqnr:>8.1f}")

## Part 5: Companding (μ-Law and A-Law)

**Problem:** Uniform quantization treats all amplitudes equally, but speech
signals spend most of their time at low amplitudes.

**Solution:** **Companding** (compress + expand) — compress the signal before
quantization, expand after reconstruction.

**μ-law** (North America, Japan):
$$y = \frac{\ln(1 + \mu |x|)}{\ln(1 + \mu)} \cdot \text{sgn}(x), \quad \mu = 255$$

**Benefit:** Low-amplitude signals get finer quantization, improving SQNR
for quiet speech by ~24 dB compared to uniform quantization.

In [ ]:
# === Part 5: Companding ===

def mu_law_compress(x, mu=255):
    return np.sign(x) * np.log(1 + mu * np.abs(x)) / np.log(1 + mu)

def mu_law_expand(y, mu=255):
    return np.sign(y) * ((1 + mu)**np.abs(y) - 1) / mu

def a_law_compress(x, A=87.6):
    ax = np.abs(x)
    y = np.where(ax < 1/A,
                 A * ax / (1 + np.log(A)),
                 (1 + np.log(A * ax)) / (1 + np.log(A)))
    return np.sign(x) * y

x_range = np.linspace(-1, 1, 1000)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Compression curves
axes[0].plot(x_range, x_range, 'k--', linewidth=1, label='Uniform (no companding)')
for mu_val, color in [(15, 'C2'), (100, 'C1'), (255, 'C0')]:
    y_mu = mu_law_compress(x_range, mu_val)
    axes[0].plot(x_range, y_mu, color, linewidth=2, label=f'\u03bc = {mu_val}')
axes[0].plot(x_range, a_law_compress(x_range), 'C3--', linewidth=2, label='A-law (A=87.6)')
axes[0].set_xlabel('Input x', fontsize=13)
axes[0].set_ylabel('Output y', fontsize=13)
axes[0].set_title('Companding Curves', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_aspect('equal')

# SQNR comparison: uniform vs mu-law
input_levels_dB = np.arange(-50, 1, 1)  # Input level relative to full scale
n_bits = 8
L = 2**n_bits

sqnr_uniform = np.zeros(len(input_levels_dB))
sqnr_mulaw = np.zeros(len(input_levels_dB))

for idx, level_dB in enumerate(input_levels_dB):
    amplitude = 10**(level_dB / 20)
    t_test = np.linspace(0, 1, 10000)
    x_test = amplitude * np.sin(2 * np.pi * 100 * t_test)
    
    # Uniform quantization
    xq_uni, err_uni, _, _ = quantize(x_test, n_bits)
    sig_p = np.mean(x_test**2)
    err_p_uni = np.mean(err_uni**2)
    sqnr_uniform[idx] = 10 * np.log10(sig_p / (err_p_uni + 1e-20))
    
    # Mu-law: compress, quantize, expand
    x_compressed = mu_law_compress(x_test)
    xq_comp, _, _, _ = quantize(x_compressed, n_bits)
    xq_expanded = mu_law_expand(xq_comp)
    err_mu = x_test - xq_expanded
    err_p_mu = np.mean(err_mu**2)
    sqnr_mulaw[idx] = 10 * np.log10(sig_p / (err_p_mu + 1e-20))

axes[1].plot(input_levels_dB, sqnr_uniform, 'C1', linewidth=2, label='Uniform 8-bit')
axes[1].plot(input_levels_dB, sqnr_mulaw, 'C0', linewidth=2, label='\u03bc-law 8-bit (\u03bc=255)')
axes[1].set_xlabel('Input Level (dB rel. full scale)', fontsize=13)
axes[1].set_ylabel('SQNR (dB)', fontsize=13)
axes[1].set_title('SQNR: Uniform vs \u03bc-law', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_xlim(-50, 0)
axes[1].set_ylim(0, 55)

# Annotate advantage at low levels
axes[1].annotate('\u03bc-law advantage\nfor quiet speech', xy=(-35, 35),
                xytext=(-25, 15), fontsize=10, color='C0',
                arrowprops=dict(arrowstyle='->', color='C0'))

# Quantization levels visualization
levels_uniform = np.linspace(-1, 1, 17)  # 4-bit for visibility
levels_mulaw = mu_law_expand(np.linspace(-1, 1, 17))

axes[2].eventplot([levels_uniform], lineoffsets=1.5, linelengths=0.5,
                  colors='C1', label='Uniform')
axes[2].eventplot([levels_mulaw], lineoffsets=0.5, linelengths=0.5,
                  colors='C0', label='\u03bc-law')
axes[2].set_xlabel('Amplitude', fontsize=13)
axes[2].set_title('Quantization Levels (4-bit)', fontsize=14, fontweight='bold')
axes[2].set_yticks([0.5, 1.5])
axes[2].set_yticklabels(['\u03bc-law', 'Uniform'])
axes[2].set_xlim(-1.1, 1.1)
axes[2].axvspan(-0.3, 0.3, alpha=0.15, color='yellow')
axes[2].text(0, 2.1, 'More levels near zero\n(where speech lives)',
            ha='center', fontsize=9, fontstyle='italic')

plt.tight_layout()
plt.show()

print("Companding Summary:")
print(f"  \u03bc-law (\u03bc=255): Used in North America & Japan")
print(f"  A-law (A=87.6):  Used in Europe")
print(f"  Both achieve ~{sqnr_mulaw[-10]:.0f} dB SQNR at low input levels with 8-bit quantization")
print(f"  Uniform 8-bit SQNR at -30 dB input: {sqnr_uniform[20]:.0f} dB")
print(f"  \u03bc-law 8-bit SQNR at -30 dB input:  {sqnr_mulaw[20]:.0f} dB")
print(f"  Improvement: ~{sqnr_mulaw[20] - sqnr_uniform[20]:.0f} dB for quiet signals")

## Part 6: Complete PCM System

**Pulse Code Modulation (PCM)** is the standard method for digitizing
analog signals. The complete pipeline:

1. **Anti-aliasing filter** (LPF, cutoff $\leq f_s/2$)
2. **Sample** at rate $f_s \geq 2B$
3. **Quantize** to $L = 2^b$ levels
4. **Encode** as binary ($b$ bits per sample)

**Bit rate:** $R_b = f_s \times b$ bits/sec

**Example (telephone):** $f_s = 8$ kHz, $b = 8$ bits $\Rightarrow R_b = 64$ kbps (DS0)

In [ ]:
# === Part 6: Complete PCM Pipeline ===

# Simulate a speech-like signal (sum of harmonics)
fs_pcm = 8000    # Telephone sampling rate
n_bits_pcm = 8   # Telephone bit depth
duration = 0.02  # 20 ms segment
t_pcm = np.arange(0, duration, 1/fs_pcm)

# Analog signal (simulated at high rate)
fs_analog = 100000
t_analog = np.arange(0, duration, 1/fs_analog)

# Speech-like signal: fundamental + harmonics
f0 = 200  # Fundamental frequency
x_analog = (0.5 * np.sin(2 * np.pi * f0 * t_analog)
          + 0.3 * np.sin(2 * np.pi * 2*f0 * t_analog)
          + 0.15 * np.sin(2 * np.pi * 3*f0 * t_analog)
          + 0.1 * np.sin(2 * np.pi * 5*f0 * t_analog)
          + 0.05 * np.sin(2 * np.pi * 4200 * t_analog))  # Near Nyquist

# Step 1: Anti-aliasing filter
nyq = fs_analog / 2
b_aa, a_aa = signal.butter(6, 3400 / nyq, btype='low')
x_filtered = signal.filtfilt(b_aa, a_aa, x_analog)

# Step 2: Sample
sample_indices = np.round(t_pcm * fs_analog).astype(int)
sample_indices = np.clip(sample_indices, 0, len(x_filtered) - 1)
x_sampled = x_filtered[sample_indices]

# Step 3: Quantize (with mu-law companding)
x_compressed = mu_law_compress(x_sampled / np.max(np.abs(x_sampled)))
x_quantized, q_err, delta_pcm, L_pcm = quantize(x_compressed, n_bits_pcm)

# Step 4: Encode to binary
# Map quantized values to integer codes
codes = np.round((x_quantized + 1 - delta_pcm/2) / delta_pcm).astype(int)
codes = np.clip(codes, 0, L_pcm - 1)

# Decode: expand
x_decoded = mu_law_expand(x_quantized) * np.max(np.abs(x_sampled))

fig, axes = plt.subplots(4, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [1.5, 1, 1, 0.8]})

# Original + filtered + sampled
axes[0].plot(t_analog * 1000, x_analog, 'C0', linewidth=1, alpha=0.4, label='Original')
axes[0].plot(t_analog * 1000, x_filtered, 'C2', linewidth=1.5, label='Anti-alias filtered')
axes[0].stem(t_pcm * 1000, x_sampled, linefmt='C1-', markerfmt='C1o', basefmt='C1-',
             label=f'Sampled ({fs_pcm/1000:.0f} kHz)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('PCM Pipeline: Telephone System (8 kHz, 8-bit, \u03bc-law)',
                  fontsize=14, fontweight='bold')
axes[0].legend(fontsize=9, loc='upper right')

# Compressed + quantized
axes[1].stem(t_pcm * 1000, x_compressed, linefmt='C3-', markerfmt='C3o', basefmt='C3-',
             label='\u03bc-law compressed')
axes[1].step(t_pcm * 1000, x_quantized, 'C4', linewidth=2, where='mid', label='Quantized')
axes[1].set_ylabel('Compressed')
axes[1].legend(fontsize=9, loc='upper right')

# Decoded output
axes[2].plot(t_analog * 1000, x_filtered, 'C2', linewidth=1, alpha=0.4, label='Original (filtered)')
axes[2].stem(t_pcm * 1000, x_decoded, linefmt='C0-', markerfmt='C0o', basefmt='C0-',
             label='Decoded')
axes[2].set_ylabel('Decoded')
axes[2].legend(fontsize=9, loc='upper right')

# Binary codes (first 8 samples)
show_n = min(8, len(codes))
binary_str = '  '.join([format(c, f'0{n_bits_pcm}b') for c in codes[:show_n]])
axes[3].text(0.5, 0.5, f'Binary codes: {binary_str} ...',
            ha='center', va='center', fontsize=11, fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
            transform=axes[3].transAxes)
axes[3].set_xlim(0, duration * 1000)
axes[3].axis('off')

for ax in axes[:3]:
    ax.set_xlim(0, duration * 1000)
axes[2].set_xlabel('Time (ms)', fontsize=12)

plt.tight_layout()
plt.show()

bitrate = fs_pcm * n_bits_pcm
print(f"PCM Telephone System (DS0):")
print(f"  Voice bandwidth:    B = 4 kHz")
print(f"  Sampling rate:      f_s = {fs_pcm/1000:.0f} kHz (Nyquist rate = 8 kHz)")
print(f"  Quantization:       {n_bits_pcm} bits ({L_pcm} levels), \u03bc-law companded")
print(f"  Bit rate:           R_b = {fs_pcm} \u00d7 {n_bits_pcm} = {bitrate/1000:.0f} kbps")
print(f"  SQNR:               \u2248{6.02*n_bits_pcm + 1.76:.0f} dB (with \u03bc-law)")
print(f"\n  T1 line: 24 \u00d7 64 kbps = 1.544 Mbps")
print(f"  E1 line: 32 \u00d7 64 kbps = 2.048 Mbps")

## Summary

### Key Formulas

| Quantity | Formula |
|----------|--------|
| Nyquist rate | $f_N = 2B$ |
| Quantization step | $\Delta = (x_{max} - x_{min}) / 2^b$ |
| Quantization noise power | $\sigma_q^2 = \Delta^2 / 12$ |
| SQNR | $\approx 6.02b + 1.76$ dB |
| PCM bit rate | $R_b = f_s \times b$ bits/sec |
| \u03bc-law compression | $y = \frac{\ln(1 + \mu|x|)}{\ln(1 + \mu)} \text{sgn}(x)$ |

### Key Takeaways

1. **Nyquist theorem:** $f_s \geq 2B$ prevents aliasing
2. **Aliasing** folds high frequencies back; anti-aliasing filter is essential
3. **Each bit adds ~6 dB** of SQNR (the "6 dB per bit" rule)
4. **Companding** (μ-law / A-law) dramatically improves SQNR for low-level signals
5. **PCM** is the foundation of all digital audio and telephony
6. **DS0 = 64 kbps:** The fundamental digital telephone channel

### Next Topics
- **Lesson 27:** BER Analysis, Matched Filtering, Signal Space
- Python Lab 5 (BER Performance)
- Reading: Chapter 10